# Entrenamiento del wake word `Claudio` para NosVers

Notebook reproducible para Google Colab Pro (GPU T4).

**Objetivo**: entrenar un modelo openWakeWord que dispare cuando Angel diga "Claudio" frente al micrófono del PC de casa.

**Parámetros (del spec NosVers · BRIEF §11 + research.md §3)**
- `target_phrase = "Claudio"`
- 4000 muestras sintéticas positivas con 5 voces TTS distintas
- 8000 negativos de LibriSpeech + MUSAN
- Umbral inicial 0.6, cooldown 3s post-activación (estos se configuran luego en `nosvers_voz/config.py`, no en el modelo)

**Salida**: `claudio.onnx` (~200 KB) que se descarga al final del notebook.

**Tiempo estimado**: 75-90 min en GPU T4 de Colab Pro.

**Procedimiento de seguridad si el entrenamiento se atasca**:
- La celda 5 (generación de positivos) y la 7 (entrenamiento) son las más largas. Si una celda lleva > 30 min sin avanzar, interrumpir kernel.
- Si Colab desconecta la VM, reanudar desde la celda 6 (los datos persisten en `/content` durante la sesión).


## 0. Verificar GPU y montar Drive (opcional)

Drive sólo se monta si Angel quiere guardar el modelo persistente. Si no, se descarga al final con `files.download`.


In [ ]:
!nvidia-smi

In [ ]:
# Opcional: descomenta para persistir el modelo en Drive
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Instalar dependencias


In [ ]:
%%capture
# openWakeWord + utilidades de entrenamiento
!pip install openwakeword piper-tts==1.2.0 audiomentations==0.33.0 onnx onnxruntime onnxconverter-common
!pip install datasets soundfile torch torchaudio numpy tqdm
!pip install acoustics
# El paquete `openwakeword` incluye los scripts de entrenamiento bajo openwakeword.train
import openwakeword
import openwakeword.train
print('openwakeword OK:', openwakeword.__version__)

In [ ]:
# Descargar el feature model preentrenado de openWakeWord (Google speech embeddings)
import openwakeword.utils
openwakeword.utils.download_models()
print('feature model descargado')

## 2. Parámetros del entrenamiento


In [ ]:
from pathlib import Path

TARGET_PHRASE = 'Claudio'
TARGET_SLUG = 'claudio'

N_POSITIVES = 4000          # muestras sintéticas con varias voces
N_NEGATIVES = 8000          # mezcla LibriSpeech + MUSAN
SAMPLE_RATE = 16000

BASE = Path('/content/owk_training')
POS_DIR = BASE / 'positives'
NEG_DIR = BASE / 'negatives'
FEAT_DIR = BASE / 'features'
MODEL_DIR = BASE / 'models'
for d in (POS_DIR, NEG_DIR, FEAT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('dirs creados en', BASE)

## 3. Descargar voces de piper-tts

Se usan 5 voces distintas (3 castellano, 2 mexicano/multilenguaje) para variar timbre y entonación. Esto reduce el sesgo del modelo a una sola voz.


In [ ]:
import urllib.request, os

VOICES = [
    ('es_ES-mls_9972-low',    'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_ES/mls_9972/low/es_ES-mls_9972-low.onnx'),
    ('es_ES-davefx-medium',   'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_ES/davefx/medium/es_ES-davefx-medium.onnx'),
    ('es_ES-sharvard-medium', 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_ES/sharvard/medium/es_ES-sharvard-medium.onnx'),
    ('es_MX-claude-high',     'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_MX/claude/high/es_MX-claude-high.onnx'),
    ('es_MX-ald-medium',      'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/es/es_MX/ald/medium/es_MX-ald-medium.onnx'),
]

VOICE_DIR = BASE / 'voices'
VOICE_DIR.mkdir(exist_ok=True)
for name, onnx_url in VOICES:
    onnx_path = VOICE_DIR / f'{name}.onnx'
    json_path = VOICE_DIR / f'{name}.onnx.json'
    if not onnx_path.exists():
        print('descargando', name)
        urllib.request.urlretrieve(onnx_url, onnx_path)
        urllib.request.urlretrieve(onnx_url + '.json', json_path)
print('voces descargadas:', sorted(p.name for p in VOICE_DIR.glob('*.onnx')))

## 4. Generar variantes de la frase para diversidad

Para hacer el modelo más robusto se generan muestras de la palabra `Claudio` en distintos contextos cortos: aislada, con prefijo ("oye Claudio", "hola Claudio"), con tono pregunta, etc.


In [ ]:
import random

PHRASE_VARIANTS = [
    'Claudio.',
    'Claudio',
    'Oye, Claudio.',
    'Hola Claudio.',
    'Claudio, ¿estás ahí?',
    'Claudio, escucha.',
    'Claudio, una pregunta.',
    'Eh, Claudio.',
    'A ver, Claudio.',
    'Claudio, atento.',
]

# 4000 muestras / 5 voces = 800 muestras por voz, repartidas entre las variantes
per_voice = N_POSITIVES // len(VOICES)
print(f'{per_voice} muestras por voz, {len(PHRASE_VARIANTS)} variantes')

## 5. Sintetizar positivos con piper

Genera los WAV con cada voz, randomizando ligeramente velocidad y silencios para evitar overfit. Ésta es la celda más larga (~15-20 min).


In [ ]:
from piper import PiperVoice
import wave, numpy as np, io
from tqdm.auto import tqdm

random.seed(42)

voices_loaded = []
for name, _ in VOICES:
    onnx_path = str(VOICE_DIR / f'{name}.onnx')
    voices_loaded.append((name, PiperVoice.load(onnx_path)))

count = 0
with tqdm(total=N_POSITIVES, desc='positivos') as pbar:
    for vname, pv in voices_loaded:
        for i in range(per_voice):
            phrase = random.choice(PHRASE_VARIANTS)
            length_scale = random.uniform(0.85, 1.15)   # velocidad ±15 %
            noise_scale  = random.uniform(0.55, 0.75)   # expresividad
            buf = io.BytesIO()
            with wave.open(buf, 'wb') as wf:
                wf.setnchannels(1)
                wf.setsampwidth(2)
                wf.setframerate(pv.config.sample_rate)
                pv.synthesize(phrase, wf, length_scale=length_scale, noise_scale=noise_scale)
            out = POS_DIR / f'{vname}_{i:04d}.wav'
            out.write_bytes(buf.getvalue())
            count += 1
            pbar.update(1)
print('positivos generados:', count)

## 6. Descargar y preparar negativos (LibriSpeech + MUSAN)

Se usan `datasets` de HuggingFace para obtener subconjuntos manejables. Cualquier audio que **no** contenga "Claudio" sirve.


In [ ]:
# LibriSpeech: clean speech (inglés, pero válido como negativo porque entrena al modelo a ignorar habla genérica)
# MUSAN: música + ruido ambiente + speech.
# Tirando del subset clean-100 de LibriSpeech y un slice de MUSAN basta.
import os, tarfile, urllib.request

# LibriSpeech dev-clean (~340MB) — ligero, suficiente como pool de negativos
LS_TAR = BASE / 'librispeech_dev_clean.tar.gz'
if not LS_TAR.exists():
    print('descargando LibriSpeech dev-clean…')
    urllib.request.urlretrieve('https://www.openslr.org/resources/12/dev-clean.tar.gz', LS_TAR)
LS_DIR = BASE / 'LibriSpeech'
if not LS_DIR.exists():
    with tarfile.open(LS_TAR) as t:
        t.extractall(BASE)
    print('extraído LibriSpeech')

# MUSAN free music + noise (~11GB completo → cogemos sólo /noise/ que pesa ~1GB)
MUSAN_NOISE = BASE / 'musan_noise.tar.gz'
if not MUSAN_NOISE.exists():
    print('descargando MUSAN noise (~1GB)…')
    urllib.request.urlretrieve('https://www.openslr.org/resources/17/musan.tar.gz', MUSAN_NOISE)
MUSAN_DIR = BASE / 'musan'
if not MUSAN_DIR.exists():
    with tarfile.open(MUSAN_NOISE) as t:
        # extraer sólo subcarpetas noise/ y music/ (los más útiles)
        members = [m for m in t.getmembers() if m.name.startswith('musan/noise') or m.name.startswith('musan/music')]
        t.extractall(BASE, members=members)
    print('extraído MUSAN (noise+music)')

In [ ]:
# Indexar todos los .flac/.wav negativos
import glob
neg_files = sorted(glob.glob(str(LS_DIR / '**' / '*.flac'), recursive=True))
neg_files += sorted(glob.glob(str(MUSAN_DIR / '**' / '*.wav'), recursive=True))
random.seed(7)
random.shuffle(neg_files)
neg_files = neg_files[:N_NEGATIVES]
print(f'negativos seleccionados: {len(neg_files)}')

# Convertirlos a WAV 16k mono troceados a ~1.5s (formato esperado por openwakeword)
import soundfile as sf
import numpy as np
from tqdm.auto import tqdm

CHUNK_SEC = 1.5
for idx, src in enumerate(tqdm(neg_files, desc='negativos')):
    try:
        audio, sr = sf.read(src, dtype='float32', always_2d=False)
    except Exception:
        continue
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        # Resample simple por torch para evitar añadir scipy
        import torchaudio.functional as F, torch
        audio = F.resample(torch.from_numpy(audio), sr, SAMPLE_RATE).numpy()
    # extraer 1 chunk aleatorio si dura > CHUNK_SEC
    n_target = int(CHUNK_SEC * SAMPLE_RATE)
    if len(audio) > n_target:
        start = random.randint(0, len(audio) - n_target)
        audio = audio[start:start + n_target]
    elif len(audio) < n_target:
        audio = np.pad(audio, (0, n_target - len(audio)))
    sf.write(str(NEG_DIR / f'neg_{idx:05d}.wav'), audio, SAMPLE_RATE)
print('negativos preparados en', NEG_DIR)

## 7. Entrenamiento con openwakeword.train

Usa el feature extractor preentrenado de Google (`embedding_model_speech.onnx`) y entrena una pequeña CNN clasificadora encima.


In [ ]:
# Generar features y entrenar de un tiro
from openwakeword.train import generate_features, train_model

# Features para positivos
generate_features(
    input_dir=str(POS_DIR),
    output_file=str(FEAT_DIR / 'positives.npy'),
    sample_rate=SAMPLE_RATE,
)

# Features para negativos
generate_features(
    input_dir=str(NEG_DIR),
    output_file=str(FEAT_DIR / 'negatives.npy'),
    sample_rate=SAMPLE_RATE,
)
print('features OK:', list(FEAT_DIR.glob('*.npy')))

In [ ]:
# Entrenamiento del clasificador
train_model(
    positive_features=str(FEAT_DIR / 'positives.npy'),
    negative_features=str(FEAT_DIR / 'negatives.npy'),
    model_name=TARGET_SLUG,
    output_dir=str(MODEL_DIR),
    target_false_positives_per_hour=0.5,   # objetivo aún más estricto que SC-006
    n_epochs=50,
    batch_size=128,
    learning_rate=1e-3,
    validation_split=0.1,
)

## 8. Validar el modelo entrenado

Carga el `.onnx` recién generado y prueba sobre 10 positivos y 50 negativos aleatorios para sanity check antes de descargar.


In [ ]:
from openwakeword import Model
import soundfile as sf, numpy as np, random

onnx_path = str(MODEL_DIR / f'{TARGET_SLUG}.onnx')
print('modelo:', onnx_path)
owwModel = Model(wakeword_models=[onnx_path], inference_framework='onnx')

def score(path):
    audio, sr = sf.read(path, dtype='float32')
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16000:
        import torchaudio.functional as F, torch
        audio = F.resample(torch.from_numpy(audio), sr, 16000).numpy()
    # OWW espera int16 / chunks de 1280 samples
    audio_int = (audio * 32767).astype(np.int16)
    max_p = 0.0
    for start in range(0, len(audio_int) - 1280, 1280):
        chunk = audio_int[start:start + 1280]
        scores = owwModel.predict(chunk)
        for _, v in scores.items():
            max_p = max(max_p, v)
    return max_p

pos_sample = random.sample(sorted(POS_DIR.glob('*.wav')), 10)
neg_sample = random.sample(sorted(NEG_DIR.glob('*.wav')), 50)
pos_scores = [score(str(p)) for p in pos_sample]
neg_scores = [score(str(p)) for p in neg_sample]
print(f'positivos: media={np.mean(pos_scores):.3f}  min={min(pos_scores):.3f}')
print(f'negativos: media={np.mean(neg_scores):.3f}  max={max(neg_scores):.3f}')
print('   → buscamos positivos cerca de 1.0 y negativos casi siempre < 0.3')

## 9. Descargar `claudio.onnx`

El archivo descargado tiene que copiarse a `~/nosvers-voz-linux/models/claudio.onnx` en el PC de casa de Angel.


In [ ]:
from google.colab import files
files.download(str(MODEL_DIR / f'{TARGET_SLUG}.onnx'))

---
## Notas de mantenimiento

- Si los falsos positivos en uso real superan 1/h (SC-006), aumentar `N_NEGATIVES` (más MUSAN noise) y volver a entrenar.
- Si Angel reporta que no se activa con su voz natural en condiciones reales, añadir 2-3 voces masculinas más en `VOICES` y regenerar.
- El umbral runtime se afina en `nosvers_voz/config.py` (`umbral_wake`). No es necesario reentrenar para subir/bajar sensibilidad.